# PCA — companion notebook

Runnable companion for [`pca.md`](./pca.md).

1. 2D point cloud with overlaid principal axes — see the geometry.
2. SVD route vs. eigendecomposition of the covariance — confirm they agree.
3. 3D anisotropic Gaussian → project to 2D, scree plot of explained variance.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(0)

## 1. 2D anisotropic cloud with principal axes

Sample from a Gaussian with covariance $\Sigma = R\,\mathrm{diag}(4, 0.5)\,R^\top$ (rotated by 30°).

In [ ]:
theta = np.deg2rad(30.0)
R = np.array([[np.cos(theta), -np.sin(theta)],
              [np.sin(theta),  np.cos(theta)]])
Sigma = R @ np.diag([4.0, 0.5]) @ R.T
X = rng.multivariate_normal(mean=[1.0, -0.5], cov=Sigma, size=400)

# Center and decompose via SVD.
mu = X.mean(axis=0)
Xc = X - mu
U, s, Vt = np.linalg.svd(Xc, full_matrices=False)
explained_var = s**2 / (len(X) - 1)
print('eigvals via SVD :', explained_var)
print('eigvals via cov :', np.linalg.eigvalsh(np.cov(X, rowvar=False))[::-1])
print('principal directions (rows = PCs):\n', Vt)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(X[:, 0], X[:, 1], s=10, alpha=0.4, label='data')

# Plot principal axes scaled by sqrt of eigenvalue (1 std).
colors = ['tab:red', 'tab:green']
for j in range(2):
    v = Vt[j] * np.sqrt(explained_var[j])
    ax.annotate('', xy=mu + 2 * v, xytext=mu,
                arrowprops=dict(arrowstyle='->', color=colors[j], lw=2.5))
    ax.text(*(mu + 2.2 * v), f'PC{j+1} (var={explained_var[j]:.2f})', color=colors[j])

ax.set_aspect('equal'); ax.legend(); ax.set_title('PCA finds the axes of the data ellipsoid')
ax.axhline(0, color='gray', lw=0.5); ax.axvline(0, color='gray', lw=0.5)
plt.show()

## 2. 3D Gaussian → 2D projection + scree plot

Build a covariance with eigenvalues $(9, 2, 0.2)$ — variance is dominated by the first two directions, so projecting onto the top 2 PCs loses only ~2% of the variance.

In [ ]:
# Random orthonormal basis Q, then build covariance.
Q, _ = np.linalg.qr(rng.standard_normal((3, 3)))
eigs = np.array([9.0, 2.0, 0.2])
Cov = Q @ np.diag(eigs) @ Q.T
X3 = rng.multivariate_normal(np.zeros(3), Cov, size=500)

pca = PCA(n_components=3).fit(X3)
print('sklearn explained variance       :', pca.explained_variance_)
print('sklearn explained variance ratio :', pca.explained_variance_ratio_)

Z = pca.transform(X3)  # scores in PC coordinates

In [ ]:
fig = plt.figure(figsize=(12, 4))

ax1 = fig.add_subplot(1, 3, 1, projection='3d')
ax1.scatter(X3[:, 0], X3[:, 1], X3[:, 2], s=6, alpha=0.4)
ax1.set_title('Original 3D data')
ax1.set_xlabel('x1'); ax1.set_ylabel('x2'); ax1.set_zlabel('x3')

ax2 = fig.add_subplot(1, 3, 2)
ax2.scatter(Z[:, 0], Z[:, 1], s=8, alpha=0.5)
ax2.set_aspect('equal')
ax2.set_xlabel('PC1'); ax2.set_ylabel('PC2')
ax2.set_title(f'Projection onto top 2 PCs\n({pca.explained_variance_ratio_[:2].sum()*100:.1f}% variance retained)')

ax3 = fig.add_subplot(1, 3, 3)
ks = np.arange(1, 4)
ax3.bar(ks, pca.explained_variance_ratio_, alpha=0.7, label='per component')
ax3.plot(ks, np.cumsum(pca.explained_variance_ratio_), 'o-', color='tab:red', label='cumulative')
ax3.set_xticks(ks); ax3.set_xlabel('PC index')
ax3.set_ylabel('explained variance ratio')
ax3.set_title('Scree plot')
ax3.legend()
plt.tight_layout(); plt.show()

## 3. Reconstruction error — Eckart–Young in action

In [ ]:
Xc3 = X3 - X3.mean(axis=0)
U, s, Vt = np.linalg.svd(Xc3, full_matrices=False)

for k in [1, 2, 3]:
    approx = U[:, :k] @ np.diag(s[:k]) @ Vt[:k, :]
    err_emp = np.linalg.norm(Xc3 - approx, 'fro')**2
    err_theory = (s[k:]**2).sum()
    print(f'k={k}: ||X - X_k||_F^2 = {err_emp:.4f}   sum_{{i>k}} sigma_i^2 = {err_theory:.4f}')